# 06 Quantization INT8 (CPU)

Cel: zastosowac dynamic quantization na student modelu i porownac rozmiar modelu.


In [ ]:
# Krok 1: importy i konfiguracja
# Po co to robimy: przygotowujemy biblioteki i sciezki, aby wykonac quantization modelu na CPU.

# Importujemy os do pracy ze sciezkami i katalogami.
import os
# Importujemy PyTorch, bo quantization wykonujemy przez API torch.
import torch
# Importujemy klase modelu klasyfikacyjnego z Transformers.
from transformers import AutoModelForSequenceClassification

# Ustalamy katalog modelu distilled (to nasz preferowany model FP32).
DISTILLED_DIR = "../artifacts/student_fp32_distilled"
# Ustalamy katalog modelu baseline jako fallback.
BASELINE_DIR = "../artifacts/student_fp32_baseline"
# Ustalamy katalog zapisu modelu INT8.
INT8_DIR = "../artifacts/student_int8"
# Tworzymy katalog INT8, jesli jeszcze nie istnieje.
os.makedirs(INT8_DIR, exist_ok=True)


In [ ]:
# Krok 2: ladowanie modelu FP32 (preferuj distilled, fallback baseline)
# Po co to robimy: quantyzujemy model FP32, wiec najpierw wybieramy najlepsze dostepne zrodlo.

# Jesli istnieje distilled model, wybieramy go; w przeciwnym razie bierzemy baseline.
source_dir = DISTILLED_DIR if os.path.exists(os.path.join(DISTILLED_DIR, "config.json")) else BASELINE_DIR
# Wczytujemy model FP32 z wybranego katalogu.
model_fp32 = AutoModelForSequenceClassification.from_pretrained(source_dir)
# Ustawiamy model w tryb ewaluacji, bo nie trenujemy go dalej.
model_fp32.eval()
# Wypisujemy, ktory katalog zostal uzyty jako zrodlo.
print("Model zrodlowy:", source_dir)


In [ ]:
# Krok 3: dynamic quantization warstw Linear do INT8
# Po co to robimy: zmniejszamy model i przyspieszamy inferencje CPU kosztem malego ryzyka spadku jakosci.

# Wykonujemy dynamic quantization: warstwy Linear przechodza z FP32 do INT8.
model_int8 = torch.quantization.quantize_dynamic(
    # Przekazujemy model FP32 jako zrodlo quantization.
    model_fp32,
    # Wskazujemy, ze quantyzujemy warstwy Linear.
    {torch.nn.Linear},
    # Ustawiamy docelowy typ wag na qint8.
    dtype=torch.qint8,
)

# Krok 4: zapisujemy model INT8 jako pelny obiekt i state_dict
# Po co to robimy: mamy wygodny plik do szybkiego ladowania i osobny plik do porownania rozmiaru.

# Zapisujemy pelny obiekt modelu INT8.
torch.save(model_int8, os.path.join(INT8_DIR, "model_int8_full.pt"))
# Zapisujemy same wagi modelu INT8.
torch.save(model_int8.state_dict(), os.path.join(INT8_DIR, "model_int8_state_dict.pt"))
# Potwierdzamy zapis modelu INT8.
print("Zapisano model INT8")


In [ ]:
# Krok 5: porownanie rozmiaru plikow
# Po co to robimy: mierzymy zysk pamieci po quantization, bo to kluczowy efekt produkcyjny.

# Ustalamy domyslna sciezke pliku FP32 w formacie safetensors.
fp32_file = os.path.join(source_dir, "model.safetensors")
# Jesli safetensors nie istnieje, uzywamy alternatywnego pliku .bin.
if not os.path.exists(fp32_file):
    # Przelaczamy sciezke na klasyczny plik pytorch_model.bin.
    fp32_file = os.path.join(source_dir, "pytorch_model.bin")

# Ustalamy sciezke pliku wag modelu INT8.
int8_file = os.path.join(INT8_DIR, "model_int8_state_dict.pt")

# Liczymy rozmiar modelu FP32 w MB.
fp32_mb = os.path.getsize(fp32_file) / (1024 * 1024)
# Liczymy rozmiar modelu INT8 w MB.
int8_mb = os.path.getsize(int8_file) / (1024 * 1024)
# Liczymy relacje rozmiaru INT8 do FP32 (im mniej, tym lepiej).
ratio = int8_mb / fp32_mb if fp32_mb > 0 else 0.0

# Wypisujemy rozmiar modelu FP32.
print(f"FP32 size [MB]: {fp32_mb:.2f}")
# Wypisujemy rozmiar modelu INT8.
print(f"INT8 size [MB]: {int8_mb:.2f}")
# Wypisujemy relacje INT8/FP32.
print(f"INT8/FP32 ratio: {ratio:.2f}")
